# Mini SCADA — tkinter vs streamlit


⚠️ Les cellules `streamlit` ci-dessous sont écrites en tant que **fichiers `.py` à exécuter en dehors du notebook** avec la commande :
```bash
streamlit run fichier.py
```
Streamlit ne s'exécute pas directement dans une cellule Jupyter classique — chaque exemple sera donc écrit dans un fichier `.py` via `%%writefile`.

## 1. Créer la fenêtre / l'application

### tkinter (rappel)

In [ ]:
import tkinter as tk

fenetre = tk.Tk()
fenetre.title("Mini SCADA")
fenetre.geometry("300x200")

# ... tout le contenu ici ...

fenetre.mainloop()   # obligatoire : lance la boucle qui garde la fenêtre ouverte

### streamlit (nouveau)

On écrit le code dans un fichier `app_streamlit.py` avec `%%writefile`, puis on le lance depuis un terminal avec `streamlit run app_streamlit.py`.

In [ ]:
%%writefile app_streamlit.py
import streamlit as st

st.title("Mini SCADA")
# ... tout le contenu ici ...
# pas de mainloop() ! le script se relance tout seul a chaque interaction

👉 **Différence clé** : tkinter tourne dans une boucle infinie (`mainloop`) qui attend des événements (clics...). Streamlit n'a pas de boucle : il **ré-exécute tout le fichier de haut en bas** à chaque interaction avec un widget.

## 2. Afficher du texte

### tkinter

In [ ]:
label = tk.Label(fenetre, text="Temperature : 55°C", font=("Arial", 14))
label.pack()   # .pack() = obligatoire pour que le widget s'affiche

### streamlit

In [ ]:
%%writefile -a app_streamlit.py

st.write("Temperature : 55°C")
# ou plus adapte pour une mesure :
st.metric("Temperature", "55 °C")

👉 **Différence clé** : en tkinter, chaque widget doit être « placé » avec `.pack()`, `.grid()` ou `.place()`. En streamlit, on écrit juste `st.xxx(...)` et ça s'affiche automatiquement, dans l'ordre du code.

## 3. Un bouton

### tkinter

In [ ]:
def au_clic():
    print("Bouton clique")

bouton = tk.Button(fenetre, text="Cliquer", command=au_clic)
bouton.pack()

### streamlit

In [ ]:
if st.button("Cliquer"):
    st.write("Bouton clique")

👉 **Différence clé** : en tkinter, on donne une **fonction** au bouton (`command=au_clic`), appelée au clic (logique événementielle classique). En streamlit, `st.button()` retourne `True` juste après le clic — on teste ça avec un `if`.

## 4. Garder un état en mémoire (le point le plus déroutant venant de tkinter)

### tkinter — une variable Python normale suffit

In [ ]:
etat_machine = {"marche": False}

def basculer():
    etat_machine["marche"] = not etat_machine["marche"]
    label_etat.config(text="MARCHE" if etat_machine["marche"] else "ARRET")

bouton = tk.Button(fenetre, text="ON/OFF", command=basculer)
bouton.pack()
label_etat = tk.Label(fenetre, text="ARRET")
label_etat.pack()

### streamlit — il faut `st.session_state`

Une variable normale serait réinitialisée à chaque clic, car **tout le script se relance**.

In [ ]:
if "marche" not in st.session_state:
    st.session_state.marche = False

if st.button("ON/OFF"):
    st.session_state.marche = not st.session_state.marche

st.write("MARCHE" if st.session_state.marche else "ARRET")

👉 **C'est LE piège classique** quand on vient de tkinter : oublier `session_state` et se demander pourquoi la variable « reset » à chaque clic.

## 5. Exemple complet — le même Mini SCADA dans les deux versions

### Version tkinter complète

In [ ]:
import tkinter as tk

fenetre = tk.Tk()
fenetre.title("Mini SCADA")

etat = {"marche": False}

def basculer():
    etat["marche"] = not etat["marche"]
    if etat["marche"]:
        bouton.config(text="MARCHE", bg="green")
    else:
        bouton.config(text="ARRET", bg="red")

tk.Label(fenetre, text="Temperature : 55°C", font=("Arial", 14)).pack(pady=10)
bouton = tk.Button(fenetre, text="ARRET", bg="red", fg="white", command=basculer)
bouton.pack(pady=10)

fenetre.mainloop()

### Version streamlit complète

Cette cellule écrit le fichier final `mini_scada_streamlit.py`. Pour le tester : ouvre un terminal dans le même dossier que ce notebook et lance :
```bash
streamlit run mini_scada_streamlit.py
```

In [ ]:
%%writefile mini_scada_streamlit.py
import streamlit as st

st.title("Mini SCADA")

if "marche" not in st.session_state:
    st.session_state.marche = False

st.metric("Temperature", "55 °C")

if st.button("Basculer ON/OFF"):
    st.session_state.marche = not st.session_state.marche

if st.session_state.marche:
    st.success("Machine en MARCHE")
else:
    st.error("Machine a l'ARRET")

## 🧠 Tableau récap à garder sous la main

| Concept | tkinter | streamlit |
|---|---|---|
| Lancer l'appli | `python fichier.py` | `streamlit run fichier.py` |
| Boucle principale | `fenetre.mainloop()` | pas de boucle, réexécution auto |
| Afficher un widget | `.pack()` / `.grid()` obligatoire | s'affiche tout seul dans l'ordre |
| Réagir à un clic | fonction passée en `command=` | `if st.button(...):` |
| Garder une valeur en mémoire | variable Python classique | `st.session_state` |
| Modifier un widget existant | `.config(text=..., bg=...)` | on ré-affiche avec une nouvelle valeur |

## 1. Les colonnes (`st.columns`) — organiser l'affichage

Au lieu d'empiler température, pression, vitesse les unes en dessous des autres, on les met côte à côte.

`st.columns(3)` retourne 3 « zones » dans lesquelles on peut écrire n'importe quel élément streamlit (`metric`, `write`, `button`...), comme des mini-conteneurs séparés.

In [ ]:
col1, col2, col3 = st.columns(3)

col1.metric("Temperature", "55 °C")
col2.metric("Pression", "4.2 bar")
col3.metric("Vitesse", "1200 rpm")

## 2. Couleurs de statut dynamiques

L'idée : la couleur/le message dépend de la **valeur réelle**, pas écrite en dur.

C'est un simple `if / elif / else` — la seule nouveauté est de choisir la fonction streamlit (`success` / `warning` / `error`) selon le cas.

In [ ]:
temperature = 68  # viendra du CSV ensuite

if temperature > 65:
    st.error(f"ALARME : temperature trop elevee ({temperature} degres C)")
elif temperature > 55:
    st.warning(f"Attention : temperature elevee ({temperature} degres C)")
else:
    st.success(f"Temperature normale ({temperature} degres C)")

## 3. Brancher les vraies données du CSV

On relie ici à ce que tu as appris en pandas.

In [ ]:
import pandas as pd

df = pd.read_csv("automatisme_donnees.csv", parse_dates=["timestamp"])

# On prend la derniere mesure connue d'une machine
derniere_mesure = df[df["machine_id"] == "Four_D"].iloc[-1]

temperature = derniere_mesure["temperature_C"]
pression = derniere_mesure["pression_bar"]
statut = derniere_mesure["statut"]

## 4. Auto-refresh (simuler du temps réel)

Streamlit ne se relance pas tout seul en continu — il faut le lui dire explicitement. La manière la plus simple et fiable est d'utiliser le composant dédié `streamlit_autorefresh`.

In [ ]:
from streamlit_autorefresh import st_autorefresh

# relance le script toutes les 5 secondes (5000 ms)
st_autorefresh(interval=5000, key="refresh")

## 5. Assemblage complet — le Mini SCADA enrichi

Cette cellule écrit le fichier final `mini_scada_enrichi.py`. Place-le dans le même dossier que `automatisme_donnees.csv`, puis lance :
```bash
streamlit run mini_scada_enrichi.py
```

In [ ]:
import streamlit as st
import pandas as pd
from streamlit_autorefresh import st_autorefresh

# ---------------------------------------------------------------------
# CONFIGURATION DE LA PAGE
# ---------------------------------------------------------------------
st.set_page_config(page_title="SCADA", page_icon="factory", layout="wide")
st.title("Supervision atelier")

# ---------------------------------------------------------------------
# AUTO-REFRESH : relance le script toutes les 5 secondes (5000 ms)
# ---------------------------------------------------------------------
st_autorefresh(interval=5000, key="refresh")

# ---------------------------------------------------------------------
# CHOIX DE LA MACHINE A SUPERVISER
# ---------------------------------------------------------------------
df = pd.read_csv("automatisme_donnees.csv", parse_dates=["timestamp"])

machines_disponibles = df["machine_id"].unique()
machine_choisie = st.selectbox("Choisir une machine", machines_disponibles)

sub = df[df["machine_id"] == machine_choisie]
derniere = sub.iloc[-1]   # derniere mesure connue pour cette machine

temperature = derniere["temperature_C"]
pression = derniere["pression_bar"]
vitesse = derniere["vitesse_rpm"]
statut = derniere["statut"]
horodatage = derniere["timestamp"]

st.caption(f"Derniere mesure : {horodatage}")

# ---------------------------------------------------------------------
# AFFICHAGE EN COLONNES
# ---------------------------------------------------------------------
col1, col2, col3 = st.columns(3)
col1.metric("Temperature", f"{temperature:.1f} degres C")
col2.metric("Pression", f"{pression:.2f} bar")
col3.metric("Vitesse", f"{vitesse:.0f} rpm")

# ---------------------------------------------------------------------
# STATUT AVEC COULEUR DYNAMIQUE (selon la temperature)
# ---------------------------------------------------------------------
if pd.isna(temperature):
    st.warning("Donnee temperature manquante pour cette mesure")
elif temperature > 65:
    st.error(f"ALARME : temperature trop elevee ({temperature:.1f} degres C)")
elif temperature > 58:
    st.warning(f"Attention : temperature elevee ({temperature:.1f} degres C)")
else:
    st.success(f"Temperature normale ({temperature:.1f} degres C)")

# ---------------------------------------------------------------------
# STATUT MACHINE (issu directement de la colonne "statut" du CSV)
# ---------------------------------------------------------------------
if statut == "ALARME":
    st.error(f"Statut machine : {statut}")
elif statut == "MAINTENANCE":
    st.warning(f"Statut machine : {statut}")
elif statut == "ARRET":
    st.info(f"Statut machine : {statut}")
else:
    st.success(f"Statut machine : {statut}")

# ---------------------------------------------------------------------
# HISTORIQUE - petit graphique de la temperature de cette machine
# ---------------------------------------------------------------------
st.subheader("Historique temperature")
st.line_chart(sub.set_index("timestamp")["temperature_C"])

## 🧠 Récap des nouveautés

| Nouveauté | Instruction clé | Rôle |
|---|---|---|
| Colonnes | `st.columns(3)` | affiche température/pression/vitesse côte à côte |
| Sélecteur de machine | `st.selectbox()` | menu déroulant pour choisir la machine à surveiller |
| Couleur dynamique | `if temperature > 65: st.error(...)` | le message change selon la valeur réelle lue |
| Données réelles | `pd.read_csv()` + `.iloc[-1]` | branché sur le vrai fichier au lieu de valeurs en dur |
| Auto-refresh | `st_autorefresh(interval=5000)` | relance la page toutes les 5 secondes automatiquement |
| Graphique historique | `st.line_chart()` | courbe d'évolution intégrée, sans matplotlib |

## ✅ Prochaines étapes possibles
- Ajouter un vrai bouton ON/OFF fonctionnel (retour sur `st.session_state`)
- Simuler l'arrivée de nouvelles données en direct dans le CSV pour voir l'auto-refresh en action
- Brancher directement sur une lecture Modbus/série au lieu du CSV statique

# Lancement du Fichiés

In [ ]:
python -m streamlit run "C:\Users\harou\Desktop\master\M2 Robotique industrielle\python\lesson_4_stremlite.py"


# Lire le fichier Excel avec pandas

In [2]:
import pandas as pd

df_ventes = pd.read_excel("achats_ventes_atelier.xlsx", sheet_name="Ventes")
print(df_ventes.head())

  N_Facture        Date            Client               Article  Quantite  \
0    V-1001  2026-08-01    Atelier Dupont   Capteur de pression         5   
1    V-1002  2026-08-02  Menuiserie Leroy  Variateur de vitesse         2   
2    V-1003  2026-08-03    Atelier Dupont      Automate S7-1200         1   
3    V-1004  2026-08-04      Garage Petit      Relais thermique        10   
4    V-1005  2026-08-05  Menuiserie Leroy    Cable RS-485 (10m)         3   

   Prix_Unitaire_EUR  Total_EUR  
0               42.5      212.5  
1              210.0      420.0  
2              890.0      890.0  
3               15.3      153.0  
4               28.9       86.7  


# Créer la fenêtre principale

In [7]:
import tkinter as tk

fenetre = tk.Tk()
fenetre.title("Facturation Atelier")
fenetre.geometry("500x600")
liste_factures = df_ventes["N_Facture"].tolist()

variable_facture = tk.StringVar(fenetre)
variable_facture.set(liste_factures[0])  # valeur par defaut
def get_facture(numero):
    ligne = df_ventes[df_ventes["N_Facture"] == numero].iloc[0]
    return ligne
def afficher_facture():
    ligne = get_facture(variable_facture.get())
    
    texte = f"""
    FACTURE N° {ligne['N_Facture']}
    Date : {ligne['Date']}
    Client : {ligne['Client']}
    
    Article : {ligne['Article']}
    Quantité : {ligne['Quantite']}
    Prix unitaire : {ligne['Prix_Unitaire_EUR']:.2f} €
    
    TOTAL : {ligne['Total_EUR']:.2f} €
    """
    
    zone_texte.config(state="normal")
    zone_texte.delete("1.0", tk.END)
    zone_texte.insert(tk.END, texte)
    zone_texte.config(state="disabled")
bouton = tk.Button(fenetre, text="Afficher la facture", command=afficher_facture)
bouton.pack(pady=10)

zone_texte = tk.Text(fenetre, width=50, height=15, font=("Courier", 10))
zone_texte.pack(pady=10)
menu = tk.OptionMenu(fenetre, variable_facture, *liste_factures)
menu.pack(pady=10)
fenetre.mainloop()